# Bronze — S&P 500 tracker prices

`landing.index_prices_raw` → `bronze.index_prices`. Every column cast to STRING.

This is the table where the cast does real work: Landing holds yfinance's own types
(timestamps, doubles, longs) and Bronze flattens them to text like everything else.

Stays **daily**. Collapsing to monthly is a business rule and belongs in Silver.

## Column guard

The CSV sources have a fixed set of columns, but yfinance's does vary by instrument
(`Capital_Gains` shows up for some and not others). Check the real column list before
building, so a mismatch is loud rather than silent.

In [0]:
EXPECTED = [
    "ticker", "Date", "Open", "High", "Low", "Close",
    "Adj_Close", "Volume", "Dividends", "Stock_Splits",
]

actual = spark.table("`index-vs-trust-pipeline`.landing.index_prices_raw").columns

print(f"actual:   {actual}")
print(f"expected: {EXPECTED}")

missing = [c for c in EXPECTED if c not in actual]
extra = [c for c in actual if c not in EXPECTED]

if missing or extra:
    print(f"\nMISMATCH -- missing: {missing}, unexpected: {extra}")
    print("Update the CAST list in the next cell to match, then re-run.")
else:
    print("\nmatch -- safe to build")

In [0]:
%sql
CREATE OR REPLACE TABLE `index-vs-trust-pipeline`.bronze.index_prices AS
SELECT
  CAST(ticker       AS STRING) AS ticker,
  CAST(`Date`       AS STRING) AS `Date`,
  CAST(Open         AS STRING) AS Open,
  CAST(High         AS STRING) AS High,
  CAST(Low          AS STRING) AS Low,
  CAST(Close        AS STRING) AS Close,
  CAST(Adj_Close    AS STRING) AS Adj_Close,
  CAST(Volume       AS STRING) AS Volume,
  CAST(Dividends    AS STRING) AS Dividends,
  CAST(Stock_Splits AS STRING) AS Stock_Splits
FROM `index-vs-trust-pipeline`.landing.index_prices_raw;

## Verification

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.landing.index_prices_raw) AS landing_rows,
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.bronze.index_prices)      AS bronze_rows,
  (SELECT COUNT(DISTINCT ticker) FROM `index-vs-trust-pipeline`.bronze.index_prices) AS tickers;

Expect the two row counts to be **equal**, and **4** tickers.

In [0]:
%sql
DESCRIBE TABLE `index-vs-trust-pipeline`.bronze.index_prices;